In [2]:
from importlib.metadata import version

print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

torch version: 2.11.0
tiktoken version: 0.12.0


In [3]:
import tiktoken
tokenizer=tiktoken.get_encoding("gpt2")

In [5]:
integers=tokenizer.encode("Akwirw ier")
print(integers)

[33901, 86, 343, 86, 220, 959]


In [6]:
for i in integers:
    print(f"{i} -> {tokenizer.decode([i])}")

33901 -> Ak
86 -> w
343 -> ir
86 -> w
220 ->  
959 -> ier


In [7]:
#练习2.1
#练习2.1 未知单词的BPE
print(tokenizer.encode("Ak"))
print(tokenizer.encode("w"))
print(tokenizer.encode("ir"))
print(tokenizer.encode("w"))
print(tokenizer.encode(" "))
print(tokenizer.encode("ier"))

print(tokenizer.decode([33901,86,343,86,220,959]))

[33901]
[86]
[343]
[86]
[220]
[959]
Akwirw ier


In [11]:
#练习2
import tiktoken
import torch
from torch.utils.data import Dataset,DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self,txt,tokenizer,max_length,stride):
        self.input_ids=[]
        self.target_ids=[]

        token_ids=tokenizer.encode(txt,allowed_special={"<|endoftext|>"})

        for i in range(0,len(token_ids)-max_length,stride):
            input_chunk=token_ids[i:i+max_length]
            target_chunk=token_ids[i+1:i+max_length+1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)
    def __getitem__(self,idx):
            return self.input_ids[idx],self.target_ids[idx]

def create_dataloader(txt,batch_size=4,max_length=256,stride=128):
     tokenizer=tiktoken.get_encoding("gpt2") 

     dataset=GPTDatasetV1(txt,tokenizer,max_length,stride)

     dataloader=DataLoader(dataset,batch_size=batch_size)

     return dataloader

with open("the-verdict.txt",'r',encoding="utf-8") as f:
     raw_text=f.read()

tokenizer=tiktoken.get_encoding("gpt2")
encode_text=tokenizer.encode(raw_text)

    

In [13]:
dataloader=create_dataloader(raw_text,batch_size=4,max_length=2,stride=2)
for batch in dataloader:
    x,y=batch
    break
x

tensor([[  40,  367],
        [2885, 1464],
        [1807, 3619],
        [ 402,  271]])

In [14]:
dataloader=create_dataloader(raw_text,batch_size=4,max_length=8,stride=2)
for batch in dataloader:
    x,y=batch
    break
x

tensor([[   40,   367,  2885,  1464,  1807,  3619,   402,   271],
        [ 2885,  1464,  1807,  3619,   402,   271, 10899,  2138],
        [ 1807,  3619,   402,   271, 10899,  2138,   257,  7026],
        [  402,   271, 10899,  2138,   257,  7026, 15632,   438]])